In [ ]:
!pip install -q -U google-genai pillow

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import shutil
import os
import json
import time
from google import genai
from PIL import Image
from google.colab import userdata
from google.colab import drive

api_key = userdata.get('KEY')
client = genai.Client(api_key=api_key)

image_folder = '/content/drive/MyDrive/allRotatedPhotos'
output_json = '/content/drive/MyDrive/analys_masks.json'
valid_extensions = ('.jpg', '.jpeg', '.png')

try:
    image_files = [f for f in os.listdir(image_folder) if f.lower().endswith(valid_extensions)]
    print(f"\nFound {len(image_files)} images. Starting extraction...\n")
except FileNotFoundError:
    print(f"Error: Could not find the folder {image_folder}. Please check the path and make sure your Drive is mounted.")
    image_files = []

results_data = []

total_session_input_tokens = 0
total_session_output_tokens = 0

for filename in image_files:
    file_path = os.path.join(image_folder, filename)
    print(f"Processing: {filename}...")

    img = Image.open(file_path)

    prompt = """
    Extract all text from this turbocharger nameplate.
    Maintain the line structure.
    If you see serial numbers or part numbers, ensure they are exact.
    CRITICAL: The text on this nameplate consists ONLY of Latin letters, numbers, hyphens (-), and slashes (/). You are strictly forbidden from outputting any other symbols like cyrillic letters or punctuation.
    Return ONLY the extracted text, no introductory sentences.
    """

    max_retries = 3
    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model="gemini-2.5-flash",
                contents=[prompt, img]
            )
            if response.usage_metadata:
                total_session_input_tokens += response.usage_metadata.prompt_token_count
                total_session_output_tokens += response.usage_metadata.candidates_token_count

            results_data.append({
                "filename": filename,
                "extracted_text": response.text.strip()
            })
            print("  -> Success!")
            time.sleep(15)
            break

        except Exception as e:
            error_message = str(e)

            if attempt < max_retries - 1 and ('503' in error_message or '429' in error_message):
                wait_time = (2 ** attempt) * 5
                print(f"  -> Server busy. Retrying in {wait_time} seconds... (Attempt {attempt + 1} of {max_retries - 1})")
                time.sleep(wait_time)
            else:
                print(f"  -> Final Error processing {filename}: {error_message}")
                results_data.append({
                    "filename": filename,
                    "error": error_message
                })
                break

if image_files:

    local_json_path = '/content/Gemininameplate_data.json'

    print("\nSaving data to local storage...")
    with open(local_json_path, 'w', encoding='utf-8') as f:
        json.dump(results_data, f, indent=4, ensure_ascii=False)
    print(f"Success! Data saved locally to: {local_json_path}")

    print("Copying file to Google Drive...")
    try:
        shutil.copy(local_json_path, output_json)
        print(f"All done! Your JSON file is securely in your Drive at: {output_json}")
    except OSError as e:
        print(f"\n[WARNING] Drive disconnected! Could not copy to Drive automatically.")
        print(f"Error details: {e}")
        print(f"Your file is safe. Click the folder icon on the left to download '{local_json_path}' manually.")
    grand_total_tokens = total_session_input_tokens + total_session_output_tokens

    summary_text = (
        f"\n--- Token Usage Summary ---\n"
        f"Total Images Processed: {len(image_files)}\n"
        f"Total Input Tokens:     {total_session_input_tokens}\n"
        f"Total Output Tokens:    {total_session_output_tokens}\n"
        f"Grand Total Tokens:     {grand_total_tokens}\n"
    )
    print(summary_text)

In [ ]:
import shutil
import os
import json
import time
from google import genai
from PIL import Image
from google.colab import userdata
from google.colab import drive

api_key = userdata.get('KEY')
client = genai.Client(api_key=api_key)

# 3. Setup your specific Drive folders
image_folder = '/content/drive/MyDrive/allRotatedPhotos'
output_json = '/content/drive/MyDrive/analys_masks.json'
valid_extensions = ('.jpg', '.jpeg', '.png')

try:
    image_files = [f for f in os.listdir(image_folder) if f.lower().endswith(valid_extensions)]
    print(f"\nFound {len(image_files)} images. Starting extraction...\n")
except FileNotFoundError:
    print(f"Error: Could not find the folder {image_folder}. Please check the path and make sure your Drive is mounted.")
    image_files = []

results_data = []

total_session_input_tokens = 0
total_session_output_tokens = 0

extraction_prompt = """
Analyze this Garrett turbocharger nameplate and extract the information into a strict JSON format.

Follow these two rules:
1. "raw_text": Extract all visible text from the nameplate. Keep it exact, with no introductory text, descriptions, or added formatting. The text on this nameplate consists ONLY of Latin letters, numbers, hyphens (-), and slashes (/). You are strictly forbidden from outputting any other symbols like cyrillic letters or punctuation.

2. "part_number": Extract ONLY the Garrett Part Number. It must follow the format of 6 digits, a hyphen (-), and 1 to 4 digits (e.g., 758219-3).
It MUST start with the digit 4, 7, or 8. If you cannot find a matching sequence, return null.

CRITICAL: Return ONLY valid JSON. Do not include markdown backticks (```) or any other conversational text.
Expected format:
{
  "raw_text": "all the text from the plate goes here...",
  "part_number": "XXXXXX-X..."
}
"""

for filename in image_files:
    file_path = os.path.join(image_folder, filename)
    print(f"Processing: {filename}...")

    img = Image.open(file_path)

    max_retries = 3
    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model="gemini-2.5-flash",
                contents=[extraction_prompt, img],
                config={"response_mime_type": "application/json"}
            )

            if response.usage_metadata:
                total_session_input_tokens += response.usage_metadata.prompt_token_count
                total_session_output_tokens += response.usage_metadata.candidates_token_count

            try:
                parsed_response = json.loads(response.text)
            except json.JSONDecodeError:
                parsed_response = {"raw_text": response.text.strip(), "part_number": None}

            results_data.append({
                "image_name": filename,
                "part_number": parsed_response.get("part_number"),
                "raw_text": parsed_response.get("raw_text")
            })
            print("  -> Success!")

            time.sleep(15)
            break

        except Exception as e:
            error_message = str(e)
            if attempt < max_retries - 1 and ('503' in error_message or '429' in error_message):
                wait_time = (2 ** attempt) * 5
                print(f"  -> Server busy. Retrying in {wait_time} seconds... (Attempt {attempt + 1} of {max_retries - 1})")
                time.sleep(wait_time)
            else:
                print(f"  -> Final Error processing {filename}: {error_message}")
                results_data.append({
                    "image_name": filename,
                    "error": error_message
                })
                break


if image_files:
    local_json_path = '/content/Gemininameplate_data.json'

    print("\nSaving data to local storage...")
    with open(local_json_path, 'w', encoding='utf-8') as f:
        json.dump(results_data, f, indent=4, ensure_ascii=False)
    print(f"Success! Data saved locally to: {local_json_path}")
    print("Copying file to Google Drive...")
    try:
        shutil.copy(local_json_path, output_json)
        print(f"All done! Your JSON file is securely in your Drive at: {output_json}")
    except OSError as e:
        print(f"\n[WARNING] Drive disconnected! Could not copy to Drive automatically.")
        print(f"Error details: {e}")
        print(f"Your file is safe. Click the folder icon on the left to download '{local_json_path}' manually.")

    grand_total_tokens = total_session_input_tokens + total_session_output_tokens

    summary_text = (
        f"\n--- Token Usage Summary ---\n"
        f"Total Images Processed: {len(image_files)}\n"
        f"Total Input Tokens:     {total_session_input_tokens}\n"
        f"Total Output Tokens:    {total_session_output_tokens}\n"
        f"Grand Total Tokens:     {grand_total_tokens}\n"
    )
    print(summary_text)